# Homework 4

## Q1: Capacitated facility location

## b

In [4]:
import gurobipy as gp
from gurobipy import GRB

# -----------------------------
# 1. Data

I = [1, 2, 3]                 # facilities
J = ['A', 'B', 'C', 'D']     # customers

# fixed opening costs
f = {
    1: 100,
    2:  80,
    3:  90
}

# capacities
s = {
    1: 20,
    2: 15,
    3: 25
}

# customer demands
d = {
    'A': 10,
    'B': 10,
    'C': 15,
    'D': 10
}

# shipping costs c[i,j]
c = {}
c[1,'A'] = 4; c[1,'B'] = 6; c[1,'C'] = 8; c[1,'D'] = 5
c[2,'A'] = 5; c[2,'B'] = 4; c[2,'C'] = 6; c[2,'D'] = 7
c[3,'A'] = 6; c[3,'B'] = 3; c[3,'C'] = 5; c[3,'D'] = 4


# 2. Build model

model = gp.Model('CFLP')

# decision variables
y = model.addVars(I, vtype=GRB.BINARY, name='open')
x = model.addVars(I, J, lb=0.0, name='ship')

# objective: fixed cost + shipping cost
model.setObjective(
    gp.quicksum(f[i]*y[i] for i in I)
  + gp.quicksum(c[i,j]*x[i,j] for i in I for j in J),
  GRB.MINIMIZE
)

# each customer j must be fully served
for j in J:
    model.addConstr(
        gp.quicksum(x[i,j] for i in I) == d[j],
        name=f"demand_{j}"
    )

# capacity constraint: sum shipments ≤ capacity * open
for i in I:
    model.addConstr(
        gp.quicksum(x[i,j] for j in J) <= s[i] * y[i],
        name=f"cap_{i}"
    )


# 3. Solve

model.optimize()


# 4. Report results

if model.status == GRB.OPTIMAL:
    print("\n--- Optimal Solution ---")
    open_facilities = [i for i in I if y[i].X > 0.5]
    print("Facilities opened:", open_facilities)
    print("\nShipments:")
    for i in I:
        for j in J:
            qty = x[i,j].X
            if qty > 1e-6:
                print(f"  Facility {i} → Customer {j}: {qty:.0f}")
    print(f"\nTotal cost: {model.ObjVal:.2f}")
else:
    print("No optimal solution found.")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-08-19
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 15 columns and 27 nonzeros
Model fingerprint: 0x3be2d76c
Variable types: 12 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [3e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+01, 2e+01]
Presolve time: 0.00s
Presolved: 7 rows, 15 columns, 27 nonzeros
Variable types: 12 continuous, 3 integer (3 binary)
Found heuristic solution: objective 465.0000000

Root relaxation: objective 3.850000e+02, 5 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0             

## c

In [1]:
# Simple Lagrangian Relaxation for CFLP

# 1) Data
I = [1, 2, 3]                # facilities
J = ['A', 'B', 'C', 'D']     # customers

f = {1:100, 2:80,  3:90}      # opening costs
s = {1:20,  2:15,  3:25}      # capacities
d = {'A':10,'B':10,'C':15,'D':10}  # demands
c = {
    (1,'A'):4,(1,'B'):6,(1,'C'):8,(1,'D'):5,
    (2,'A'):5,(2,'B'):4,(2,'C'):6,(2,'D'):7,
    (3,'A'):6,(3,'B'):3,(3,'C'):5,(3,'D'):4
}

# 2) Subgradient setup
max_iters = 100
lam = {i: 0.0 for i in I}   # multipliers λ_i
t0  = 0.5                 # initial step size

# 3) Iterations
for k in range(1, max_iters+1):
    # ---- assignment subproblem ----
    assign = {}              # assign[j] = chosen facility
    use = {i:0 for i in I}   # total demand at facility i
    bound_x = 0.0
    for j in J:
        # pick i that minimizes c[i,j] + λ_i
        best_i = min(I, key=lambda i: c[(i,j)] + lam[i])
        assign[j] = best_i
        use[best_i] += d[j]
        bound_x += d[j] * (c[(best_i,j)] + lam[best_i])

    # ---- opening subproblem ----
    open_dec = {}
    bound_y = 0.0
    for i in I:
        # decide y_i = 1 if f_i - λ_i * s_i < 0
        if f[i] - lam[i]*s[i] < 0:
            open_dec[i] = 1
            bound_y += (f[i] - lam[i]*s[i])
        else:
            open_dec[i] = 0
            # if we don’t open, penalty = 0

    # total Lagrangian bound
    lag_bound = bound_x + bound_y

    # check if this is actually feasible
    feas = True
    for i in I:
        if use[i] > 0 and open_dec[i]==0:
            feas = False
        if use[i] > s[i]:
            feas = False

    # ---- compute subgradient and update λ ----
    grad = {}
    for i in I:
        # g_i = sum_j x_ij - s_i*y_i
        grad[i] = use[i] - s[i]*open_dec[i]

    step = t0 / k
    for i in I:
        lam[i] = max(0.0, lam[i] + step * grad[i])

    # print iteration results
    print(f"Iter {k}")
    print("  lambda =", {i: round(lam[i],2) for i in I})
    print("  assign =", assign)
    print("  open   =", open_dec)
    print("  LagrBound =", round(lag_bound,2), 
          " Feasible?" , feas)
    print("-"*40)



Iter 1
  lambda = {1: 5.0, 2: 0.0, 3: 17.5}
  assign = {'A': 1, 'B': 3, 'C': 3, 'D': 3}
  open   = {1: 0, 2: 0, 3: 0}
  LagrBound = 185.0  Feasible? False
----------------------------------------
Iter 2
  lambda = {1: 5.0, 2: 11.25, 3: 11.25}
  assign = {'A': 2, 'B': 2, 'C': 2, 'D': 2}
  open   = {1: 0, 2: 0, 3: 1}
  LagrBound = -97.5  Feasible? False
----------------------------------------
Iter 3
  lambda = {1: 12.5, 2: 8.75, 3: 7.08}
  assign = {'A': 1, 'B': 1, 'C': 1, 'D': 1}
  open   = {1: 0, 2: 1, 3: 1}
  LagrBound = 215.0  Feasible? False
----------------------------------------
Iter 4
  lambda = {1: 10.0, 2: 6.88, 3: 9.58}
  assign = {'A': 3, 'B': 3, 'C': 3, 'D': 3}
  open   = {1: 1, 2: 1, 3: 1}
  LagrBound = 235.42  Feasible? False
----------------------------------------
Iter 5
  lambda = {1: 8.0, 2: 8.88, 3: 8.08}
  assign = {'A': 2, 'B': 2, 'C': 2, 'D': 3}
  open   = {1: 1, 2: 1, 3: 1}
  LagrBound = 283.75  Feasible? False
----------------------------------------
Iter 6
  l

## Q2: Multicommodity Flow with Nonhomogeneous Goods

## a

In [24]:
import gurobipy as gp
from gurobipy import GRB

# Create model
model = gp.Model("MulticommodityFlow")

# Data definitions
commodities = ['F', 'W', 'M']
arcs = [(1,3), (1,4), (2,3), (2,5), (3,4), (3,5), (4,6), (5,6), (6,7)]

capacities = {
    (1,3): 60, (1,4): 45, (2,3): 30, (2,5): 30,
    (3,4): 45, (3,5): 30, (4,6): 30, (5,6): 30, (6,7): 60
}

costs = {
    (1,3): 3, (1,4): 2, (2,3): 4, (2,5): 1,
    (3,4): 2, (3,5): 3, (4,6): 2, (5,6): 2, (6,7): 1
}

# Decision variables: flow per commodity and arc
x = model.addVars(commodities, arcs, name="x", lb=0)

# Objective: minimize total shipping cost
model.setObjective(
    gp.quicksum(costs[i,j] * x[k,i,j] for k in commodities for (i,j) in arcs),
    GRB.MINIMIZE
)

# Supply constraints for nodes 1 and 2 (must send 8 and 7 units respectively per commodity)
for k in commodities:
    model.addConstr(gp.quicksum(x[k,1,j] for (i,j) in arcs if i == 1) == 8, f"supply1_{k}")
    model.addConstr(gp.quicksum(x[k,2,j] for (i,j) in arcs if i == 2) == 7, f"supply2_{k}")

# Flow conservation for intermediate nodes (3,4,5,6)
intermediate_nodes = [3,4,5,6]
for n in intermediate_nodes:
    for k in commodities:
        inflow = gp.quicksum(x[k,i,n] for (i,j) in arcs if j == n)
        outflow = gp.quicksum(x[k,n,j] for (i,j) in arcs if i == n)
        model.addConstr(inflow == outflow, f"flow_cons_{k}_{n}")

# Demand constraint at node 7 (15 units per commodity)
for k in commodities:
    model.addConstr(x[k,6,7] == 15, f"demand_{k}")

# Arc capacity constraints (sum of all commodities per arc)
for (i,j) in arcs:
    model.addConstr(gp.quicksum(x[k,i,j] for k in commodities) <= capacities[i,j], f"cap_{i}_{j}")

# Solve the model
model.optimize()

# Output the solution
if model.status == GRB.OPTIMAL:
    print(f"Total cost: {model.ObjVal}")
    for k in commodities:
        print(f"\nCommodity {k}:")
        for (i,j) in arcs:
            if x[k,i,j].X > 1e-6:  # Ignore near-zero flows
                print(f"  Arc ({i},{j}): {x[k,i,j].X:.2f}")
else:
    print("No optimal solution found.")



Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 30 rows, 27 columns and 81 nonzeros
Model fingerprint: 0xe26c625c
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 4e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [7e+00, 6e+01]
Presolve removed 30 rows and 27 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0400000e+02   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.040000000e+02
Total cost: 204.0

Commodity F:
  Arc (1,4): 8.00
  Arc (2,5): 7.00
  Arc (4,6): 8.00
  Arc (5,6): 7.00
  Arc (6,7): 15.00

Commodity W:
  Arc (1,4): 8.00
  Arc (2,5): 7.00
  Arc (4,6): 8.00
  Arc (5,6): 7.00
  Arc (6,7): 15.00

Commodity M:
  Arc 

## b

In [42]:
import gurobipy as gp
from gurobipy import GRB
import math

# ---------------- DATA -----------------
K  = ['F','W','M']
A  = [(1,3),(1,4),(2,3),(2,5),(3,4),(3,5),(4,6),(5,6),(6,7)]
cap = {(1,3):60,(1,4):45,(2,3):30,(2,5):30,
       (3,4):45,(3,5):30,(4,6):30,(5,6):30,(6,7):60}
cost= {(1,3):3,(1,4):2,(2,3):4,(2,5):1,
       (3,4):2,(3,5):3,(4,6):2,(5,6):2,(6,7):1}

bal = {(n,k):0 for n in range(1,8) for k in K}
for k in K: bal[(1,k)], bal[(2,k)], bal[(7,k)] = 8, 7, -15   # supplies/demand

# ---------- helper builds one sub‑problem ------------
def build_sub(k, lam):
    m = gp.Model(f"sub_{k}"); m.Params.OutputFlag = 0
    x = m.addVars(A, name='x', lb=0)
    m.setObjective(gp.quicksum((cost[a] + lam[a]) * x[a] for a in A), GRB.MINIMIZE)
    for n in range(1,8):
        m.addConstr(gp.quicksum(x[a] for a in A if a[0]==n) -
                    gp.quicksum(x[a] for a in A if a[1]==n) == bal[(n,k)])
    return m, x

# -------------- sub‑gradient loop --------------------
lam  = {a:0.0 for a in A}
UB   = float('inf')
best = None
tol  = 1e-6
max_it = 30
alpha = 1.8

for it in range(max_it+1):      
    # ----- solve |K| sub‑problems ----------
    flows = {a:{k:0 for k in K} for a in A}
    LB = -sum(lam[a]*cap[a] for a in A)
    for k in K:
        sub, xk = build_sub(k, lam)
        sub.optimize()
        LB += sub.ObjVal
        for a in A: flows[a][k] = xk[a].X

    # -------- heuristic primal -------------
    feas = True; primal_cost = 0
    for a in A:
        tot = sum(flows[a][k] for k in K)
        if tot > cap[a] + 1e-8: feas = False; break
        primal_cost += cost[a]*tot
    if feas and primal_cost < UB: UB, best = primal_cost, flows

    # -------- sub‑gradient -----------------
    g = {a: sum(flows[a][k] for k in K) - cap[a] for a in A}
    gap = UB - LB
    norm_g = math.sqrt(sum(v*v for v in g.values()))
    step = 0.0 if norm_g < 1e-12 else alpha*gap/(norm_g**2)


    print(f"\n--- Iteration {it} ---")
    print(f"Lagrangian Lower Bound (LB): {LB:.4f}")
    print(f"Best Lower Bound found:     {LB:.4f}")
    print(f"Upper Bound (UB):           {UB:.4f}")
    print(f"Optimality Gap (UB - best LB): {gap:.4f}")
    print("Subgradient (non‑zero elements):")
    for a in A:
        if abs(g[a])>1e-6:
            print(f"  Arc {a}: {g[a]: .4f}")
    print(f"Step Size: {step:.6f}")


    if gap <= tol: break

    # -------- update λ ---------------------
    any_change = False
    for a in A:
        new = max(0.0, lam[a] + step*g[a])
        if abs(new-lam[a])>1e-12:
            lam[a] = new; any_change = True
    print("Updated Lambda (non‑zero elements):")
    if any_change:
        for a in A:
            if lam[a]>1e-8:
                print(f"  Arc {a}: {lam[a]:.6f}")
    else:
        print("  All lambda values are close to zero.")

# -------------- final report -----------------
print("\n==== FINAL SUMMARY ====")
print(f"Best primal cost (UB): {UB:.2f}")
print(f"Best dual  bound (LB): {LB:.2f}")
print(f"Relative gap: {100*(UB-LB)/UB:.4f}%")
if best:
    for k in K:
        print(f"\nCommodity {k}:")
        for a in A:
            if best[a][k]>1e-6:
                print(f"  {best[a][k]:.0f} units on arc {a}")




--- Iteration 0 ---
Lagrangian Lower Bound (LB): 204.0000
Best Lower Bound found:     204.0000
Upper Bound (UB):           204.0000
Optimality Gap (UB - best LB): 0.0000
Subgradient (non‑zero elements):
  Arc (1, 3): -60.0000
  Arc (1, 4): -21.0000
  Arc (2, 3): -30.0000
  Arc (2, 5): -9.0000
  Arc (3, 4): -45.0000
  Arc (3, 5): -30.0000
  Arc (4, 6): -6.0000
  Arc (5, 6): -9.0000
  Arc (6, 7): -15.0000
Step Size: 0.000000

==== FINAL SUMMARY ====
Best primal cost (UB): 204.00
Best dual  bound (LB): 204.00
Relative gap: 0.0000%

Commodity F:
  8 units on arc (1, 4)
  7 units on arc (2, 5)
  8 units on arc (4, 6)
  7 units on arc (5, 6)
  15 units on arc (6, 7)

Commodity W:
  8 units on arc (1, 4)
  7 units on arc (2, 5)
  8 units on arc (4, 6)
  7 units on arc (5, 6)
  15 units on arc (6, 7)

Commodity M:
  8 units on arc (1, 4)
  7 units on arc (2, 5)
  8 units on arc (4, 6)
  7 units on arc (5, 6)
  15 units on arc (6, 7)


## c

In [80]:
import gurobipy as gp
from gurobipy import GRB
import heapq, math
from collections import Counter

# ------------ DATA -------------
K = ['F','W','M']
origins = {1: 8, 2: 7} # supply
sink = 7
A = [(1,3),(1,4),(2,3),(2,5),(3,4),(3,5),(4,6),(5,6),(6,7)]
c = {(1,3):3,(1,4):2,(2,3):4,(2,5):1,(3,4):2,(3,5):3,(4,6):2,(5,6):2,(6,7):1}
u = {(1,3):60,(1,4):45,(2,3):30,(2,5):30,(3,4):45,(3,5):30,(4,6):30,(5,6):30,(6,7):60}



def column_generation(graph_nodes, graph_arcs, arc_weights, start_node):
    dist = {node: float('inf') for node in graph_nodes}
    dist[start_node] = 0


    pred_arc = {} 
    pred_node = {}

    num_nodes = len(graph_nodes) 

    for _ in range(num_nodes - 1):
        # Create a copy to iterate over, as we might modify dist
        relaxed_an_edge_in_this_pass = False
        for (u, v) in graph_arcs:
            # Check if the source node of the arc (u) has a reachable distance.
            if dist[u] != float('inf'):
                weight = arc_weights.get((u, v), float('inf')) # Get weight, use inf if arc not in weights
                if dist[u] + weight < dist[v]:
                    dist[v] = dist[u] + weight # Update the distance to v
                    pred_node[v] = u           # Set u as the predecessor *node* of v
                    pred_arc[v] = (u,v)        # Set (u,v) as the predecessor *arc* of v
                    relaxed_an_edge_in_this_pass = True
        # Optimization: If no edge was relaxed in a pass, no further passes will relax edges either.
        if not relaxed_an_edge_in_this_pass:
             break


    for (u, v) in graph_arcs:
        if dist[u] != float('inf'):
            weight = arc_weights.get((u, v), float('inf'))
            if dist[u] + weight < dist[v]:
                return None, None # Indicate negative cycle found by returning None

    # Return the distances and the predecessor arc dictionary for path reconstruction
    return dist, pred_arc



# Reconstructs the path as a list of arcs from the predecessor arc information.
# Returns a list of arcs representing the path, or None if no path exists or issues.
def reconstruct_path_arcs(pred_arc, start_node, end_node):
    path_arcs = []
    current_node = end_node
    
    # Trace back using the predecessor arcs until we reach the start node.
    # The predecessor arc for a node 'v' is the arc (u,v) that led to the shortest path to v.
    while current_node != start_node:
        # Get the arc that leads to the current_node
        arc_leading_to_current = pred_arc.get(current_node)
    
        if arc_leading_to_current is None:
             return None # Indicate path not found or reconstruction failed.
             
        path_arcs.append(arc_leading_to_current) 
        current_node = arc_leading_to_current[0] 

    path_arcs.reverse() 
    return path_arcs


# Adds a new variable representing flow on a given path to the Gurobi RMP model
# and updates the relevant capacity and supply constraints
def add_path_var(model, cap_con_dict, sup_con_dict, path_arcs, origin_node, commodity, var_name, original_costs):
    # Calculate the original cost of the path (sum of original costs of arcs in the path).
    path_cost = sum(original_costs.get(a, 0) for a in path_arcs) # Use .get with default 0 for safety
    var = model.addVar(lb=0, obj=path_cost, name=var_name)


    for a in path_arcs:
        if a in cap_con_dict: # Ensure the arc has a capacity constraint
            model.chgCoeff(cap_con_dict[a], var, p[commodity][a])
        # else: print(f"Warning: Arc {a} used by path {var_name} is not in capacity constraints.")

    # Update the supply constraint for the path's origin node and commodity.
    if (origin_node, commodity) in sup_con_dict: # Ensure the supply constraint exists
        model.chgCoeff(sup_con_dict[(origin_node, commodity)], var, 1.0)
    # else: print(f"Warning: Supply constraint for ({origin_node},{commodity}) not found for variable {var_name}.")
    
    return var 
# --- Main Column Generation Algorithm ---

# 1. Build the initial Restricted Master Problem (RMP).
model = gp.Model("MulticommodityFlow_ColumnGeneration_RMP")
model.Params.OutputFlag = 0 # Suppress Gurobi solver output

print("Building initial RMP...")


cap_con = {} # Dictionary to store constraint objects, indexed by arc.
for a in A:
    # Initialize LHS with 0. Flow variables will be added later.
    cap_con[a] = model.addLConstr(0, GRB.LESS_EQUAL, u[a], name=f"cap_{a}")

# Create supply equality constraints: sum of flow out of origin = supply[origin]

sup_con = {} # Dictionary to store constraint objects, indexed by (origin, commodity).
for o, q in origins.items():
    for k in K:
        # Initialize LHS with 0. Flow variables will be added later.
        sup_con[(o, k)] = model.addLConstr(0, GRB.EQUAL, q, name=f"supply_{o}_{k}")

# Identify initial paths (shortest paths on original costs from each source to sink).
# These paths will form the initial columns in the RMP.
initial_paths_arcs = {} # Store initial paths as {commodity: {origin: list_of_arc_tuples}}

print("Finding initial paths (shortest paths on original costs)...")
# Use original costs for finding initial shortest paths.
initial_shortest_path_weights = c.copy()

for k in K:
    initial_paths_arcs[k] = {}
    for start_node in origins:
       
        dist, pred_arc = column_generation(nodes, A, initial_shortest_path_weights, start_node)

        # Check if a path was found to the sink and no negative cycle was detected (N/A with original costs).
        if dist is not None and dist[sink] != float('inf'):
            # Reconstruct the path as a list of arcs.
            path_arcs = reconstruct_path_arcs(pred_arc, start_node, sink)

            if path_arcs:
                initial_paths_arcs[k][start_node] = path_arcs # Store the found path arcs.
              

# Add initial path variables (columns) to the RMP based on the identified initial paths.
initial_rmp_vars = []
print("Adding initial RMP columns...")
for k in K:
    for o in origins:
        if o in initial_paths_arcs[k]:
            path_arcs = initial_paths_arcs[k][o]
            var_name = f"Path_{o}_{k}_Initial"
            # Add the path variable using the helper function.
            var = add_path_var(model, cap_con, sup_con, path_arcs, o, k, var_name, c)
            initial_rmp_vars.append(var) # Store the variable object


# Set the objective sense to minimization (already implicitly done by obj in addVar, but explicit is clear).
model.ModelSense = GRB.MINIMIZE # Correct way to set objective sense

# Update the model after adding initial variables and setting up constraints.
model.update()
print("Initial RMP built and updated.")


# --- Column Generation Loop ---
print("\nStarting Column Generation Loop...")

iteration = 0

min_reduced_cost_overall = -1.0 # Start with a negative value to enter the loop.


max_iterations = 100 


# Loop continues as long as negative reduced cost paths are found AND max iterations are not reached.

while min_reduced_cost_overall < -1e-6 and iteration < max_iterations:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")

    # Step 2: Solve the current Restricted Master Problem (RMP).
    model.optimize()

    # Check the optimization status of the RMP. If not optimal, something went wrong.
    if model.status != GRB.OPTIMAL:
        print(f"RMP not optimal. Status: {model.status}. Exiting column generation.")
        if model.status == GRB.INFEASIBLE:
             print("The RMP is infeasible. This could indicate the original problem is infeasible.")
        break # Exit the main column generation loop

    # Step 3: Get the dual variables from the RMP solution.
    # Duals for the supply equality constraints. Access using .Pi attribute.
    
    sigma = {(o,k): sup_con[(o,k)].Pi for o in origins for k in K}
    
    # Duals for the capacity. These should be non-negative.
    alpha = {a: cap_con[a].Pi for a in A} # Capacity duals

    # Print the current RMP objective value and some duals for monitoring progress.
    print(f"RMP obj = {model.ObjVal:.4f}")
    # Print duals (sigma for supplies and non-zero alpha for capacities).
    print("  sigma (supplies):", {k: (sigma[(1,k)], sigma[(2,k)]) for k in K})
    print("  alpha (arcs with non-zero duals):", {a: round(alpha[a],6) for a in A if abs(alpha[a])>1e-8}) # Using 1e-8 tolerance 
    # Step 4: Find columns (paths) with negative reduced cost.
   
    new_columns_to_add = [] 
    min_reduced_cost_overall = 0 # Reset the minimum reduced cost found in this pricing step.
                                
    print("Solving pricing subproblems...")

    # Solve a pricing problem for each origin and commodity.
    for o in origins:
        for k in K:
            # Define the modified arc costs for the shortest path problem: original cost - capacity dual.
            w = {a: c[a] - alpha[a]*p[k][a] for a in A}


            dist, pred_arc = column_generation(nodes, A, w, o)

            if dist is None:
                 print(f"  Pricing (o={o}, k={k}): Negative cycle detected!")
                 print("  The RMP is unbounded. This suggests the original problem is unbounded.")
                 min_reduced_cost_overall = -float('inf') # Set to negative infinity to signal unboundedness.
                 break # Exit pricing loops

            # Check if the sink is reachable from the start node.
            if dist[sink] != float('inf'):
                # The shortest path cost found is the sum of the modified costs along the path.
                shortest_path_modified_cost = dist[sink]

                # Calculate the reduced cost of the found path.
                # Reduced Cost = (Shortest path cost on modified arcs) - (Supply dual for commodity and origin)
                red_cost = shortest_path_modified_cost - sigma[(o,k)]

                # Print the reduced cost found for each pricing subproblem.
                print(f"  Pricing (o={o}, k={k})  rc = {red_cost:.4f}") 

                # Step 5: Check if the reduced cost is negative.
                if red_cost < -1e-6:
                    # If the reduced cost is negative, the found path is a candidate for adding as a column.

                    # Update the minimum reduced cost found across all pricing problems in this iteration.
                    # This value is used for the overall convergence check.
                    min_reduced_cost_overall = min(min_reduced_cost_overall, red_cost)

                    # Reconstruct the path as a list of arcs.
                    path_arcs = reconstruct_path_arcs(pred_arc, o, sink)

                    # Ensure the path is valid before adding (check if reconstruction was successful).
                    if path_arcs is not None:
                         # Store the data needed to add this column later.
                         new_columns_to_add.append({'commodity': k, 'origin': o, 'path_arcs': path_arcs, 'reduced_cost': red_cost})
            
                    # else: print(f"    Pricing (o={o}, k={k}): Could not reconstruct path for a negative reduced cost column.")


        if min_reduced_cost_overall == -float('inf'):
             # If unboundedness detected in pricing for any commodity/source, stop all pricing.
             break

    # Print the minimum reduced cost found across all pricing problems in this iteration.
    print(f"  Minimum reduced cost found in pricing: {min_reduced_cost_overall:.4f}")

    # Step 6: Check stopping criterion.
    # If the minimum reduced cost found in this iteration is non-negative, no beneficial columns were found, and the algorithm converges.
    if min_reduced_cost_overall >= -1e-6:
        print("\nNo column with negative reduced cost found in pricing. Optimal solution reached.")
        break # Exit the main column generation loop.

    # Step 7: Add entering columns to the RMP.
    if new_columns_to_add:
        print(f"\nAdding {len(new_columns_to_add)} new columns to RMP...")
        # Iterate through the collected new column data and add each path variable to the RMP.
        for col_data in new_columns_to_add:
            k = col_data['commodity']
            o = col_data['origin']
            path_arcs = col_data['path_arcs']
            red_cost = col_data['reduced_cost'] 
            
            # Generate a unique name for the new path variable.
            name = f"Path_{o}_{k}_iter{iteration}_rc{red_cost:.2f}".replace('-', 'm').replace('.', 'd') # Make name valid chars
            added_var = add_path_var(model, cap_con, sup_con, path_arcs, o, k, name, c)

        # Important: Update the Gurobi model after adding variables and constraints in this iteration.
        model.update()
        print("RMP updated with new columns.")



print("\n==== FINAL SOLUTION ====")


# Check the final status of the model to ensure it's optimal.
if model.status == GRB.OPTIMAL and min_reduced_cost_overall >= -1e-6:
    # Print the optimal objective value of the RMP, which is the optimal value of the original problem.
    print(f"Optimal Total Cost = {model.ObjVal:.4f}")

    print("\nNon-zero Path Flows:")

    for v in model.getVars():
        if v.X > 1e-6:
            print(f"{v.VarName:30s}: {v.X:.4f}  (cost/unit = {v.Obj:.0f})")


            

Building initial RMP...
Finding initial paths (shortest paths on original costs)...
Adding initial RMP columns...
Initial RMP built and updated.

Starting Column Generation Loop...

--- Iteration 1 ---
RMP obj = 204.0000
  sigma (supplies): {'F': (5.0, 4.0), 'W': (5.0, 4.0), 'M': (5.0, 4.0)}
  alpha (arcs with non-zero duals): {}
Solving pricing subproblems...
  Pricing (o=1, k=F)  rc = 0.0000
  Pricing (o=1, k=W)  rc = 0.0000
  Pricing (o=1, k=M)  rc = 0.0000
  Pricing (o=2, k=F)  rc = 0.0000
  Pricing (o=2, k=W)  rc = 0.0000
  Pricing (o=2, k=M)  rc = 0.0000
  Minimum reduced cost found in pricing: 0.0000

No column with negative reduced cost found in pricing. Optimal solution reached.

==== FINAL SOLUTION ====
Optimal Total Cost = 204.0000

Non-zero Path Flows:
Path_1_F_Initial              : 8.0000  (cost/unit = 5)
Path_2_F_Initial              : 7.0000  (cost/unit = 4)
Path_1_W_Initial              : 8.0000  (cost/unit = 5)
Path_2_W_Initial              : 7.0000  (cost/unit = 4)
P

## d

In [173]:
import gurobipy as gp
from gurobipy import GRB
import heapq, math
from collections import Counter

# ------------ DATA -------------
K = ['F','W','M']
origins = {1: 8, 2: 7} # supply
sink = 7
A = [(1,3),(1,4),(2,3),(2,5),(3,4),(3,5),(4,6),(5,6),(6,7)]
c = {(1,3):3,(1,4):2,(2,3):4,(2,5):1,(3,4):2,(3,5):3,(4,6):2,(5,6):2,(6,7):1}
u = {(1,3):60,(1,4):45,(2,3):30,(2,5):30,(3,4):45,(3,5):30,(4,6):30,(5,6):30,(6,7):60}

p = {
    'F': {(1,3):1,(1,4):1,(2,3):2,(2,5):1,(3,4):1,(3,5):2,(4,6):1,(5,6):1,(6,7):1},
    'W': {(1,3):2,(1,4):1,(2,3):1,(2,5):2,(3,4):1,(3,5):1,(4,6):2,(5,6):1,(6,7):1},
    'M': {(1,3):3,(1,4):2,(2,3):2,(2,5):1,(3,4):1,(3,5):3,(4,6):1,(5,6):2,(6,7):1}
}

def column_generation(graph_nodes, graph_arcs, arc_weights, start_node):
    dist = {node: float('inf') for node in graph_nodes}
    dist[start_node] = 0


    pred_arc = {} 
    pred_node = {}

    num_nodes = len(graph_nodes) 

    for _ in range(num_nodes - 1):
        # Create a copy to iterate over
        relaxed_an_edge_in_this_pass = False
        for (u, v) in graph_arcs:
            # Check if the source node of the arc (u) has a reachable distance.
            if dist[u] != float('inf'):
                weight = arc_weights.get((u, v), float('inf')) # Get weight, use inf if arc not in weights
                if dist[u] + weight < dist[v]:
                    dist[v] = dist[u] + weight # Update the distance to v
                    pred_node[v] = u           # Set u as the predecessor *node* of v
                    pred_arc[v] = (u,v)        # Set (u,v) as the predecessor *arc* of v
                    relaxed_an_edge_in_this_pass = True
        # Optimization: If no edge was relaxed in a pass, no further passes will relax edges either.
        if not relaxed_an_edge_in_this_pass:
             break


    for (u, v) in graph_arcs:
        if dist[u] != float('inf'):
            weight = arc_weights.get((u, v), float('inf'))
            if dist[u] + weight < dist[v]:
                return None, None # Indicate negative cycle found by returning None

    # Return the distances and the predecessor arc dictionary for path reconstruction
    return dist, pred_arc



# Reconstructs the path as a list of arcs from the predecessor arc information.
# Returns a list of arcs representing the path, or None if no path exists or issues.
def reconstruct_path_arcs(pred_arc, start_node, end_node):
    path_arcs = []
    current_node = end_node
    
    # Trace back using the predecessor arcs until we reach the start node.
    # The predecessor arc for a node 'v' is the arc (u,v) that led to the shortest path to v.
    while current_node != start_node:
        # Get the arc that leads to the current_node
        arc_leading_to_current = pred_arc.get(current_node)
    
        if arc_leading_to_current is None:
             return None # Indicate path not found or reconstruction failed.
             
        path_arcs.append(arc_leading_to_current) 
        current_node = arc_leading_to_current[0] 

    path_arcs.reverse() 
    return path_arcs


# Adds a new variable representing flow on a given path to the Gurobi RMP model
# and updates the relevant capacity and supply constraints

def add_path_var(model, cap_con_dict, sup_con_dict, path_arcs, origin_node, commodity, var_name, original_costs):
    # Calculate the original cost of the path (sum of original costs of arcs in the path).
    path_cost = sum(original_costs.get(a, 0) for a in path_arcs) # Use .get with default 0 for safety
    var = model.addVar(lb=0, obj=path_cost, name=var_name)


    for a in path_arcs:
        if a in cap_con_dict: # Ensure the arc has a capacity constraint
            model.chgCoeff(cap_con_dict[a], var, p[commodity][a])
        # else: print(f"Warning: Arc {a} used by path {var_name} is not in capacity constraints.")

    # Update the supply constraint for the path's origin node and commodity.
    if (origin_node, commodity) in sup_con_dict: # Ensure the supply constraint exists
        model.chgCoeff(sup_con_dict[(origin_node, commodity)], var, 1.0)
    # else: print(f"Warning: Supply constraint for ({origin_node},{commodity}) not found for variable {var_name}.")
    
    return var 
# --- Main Column Generation Algorithm ---

# 1. Build the initial Restricted Master Problem (RMP).
model = gp.Model("MulticommodityFlow_ColumnGeneration_RMP")
model.Params.OutputFlag = 0 # Suppress Gurobi solver output

print("Building initial RMP...")


cap_con = {} # Dictionary to store constraint objects, indexed by arc.
for a in A:
    # Initialize LHS with 0. Flow variables will be added later.
    cap_con[a] = model.addLConstr(0, GRB.LESS_EQUAL, u[a], name=f"cap_{a}")

# Create supply equality constraints: sum of flow out of origin = supply[origin]

sup_con = {} # Dictionary to store constraint objects, indexed by (origin, commodity).
for o, q in origins.items():
    for k in K:
        # Initialize LHS with 0. Flow variables will be added later.
        sup_con[(o, k)] = model.addLConstr(0, GRB.EQUAL, q, name=f"supply_{o}_{k}")

# Identify initial paths (shortest paths on original costs from each source to sink).
# These paths will form the initial columns in the RMP.
initial_paths_arcs = {} # Store initial paths as {commodity: {origin: list_of_arc_tuples}}

print("Finding initial paths (shortest paths on original costs)...")
# Use original costs for finding initial shortest paths.
initial_shortest_path_weights = c.copy()

for k in K:
    initial_paths_arcs[k] = {}
    for start_node in origins:
       
        dist, pred_arc = column_generation(nodes, A, initial_shortest_path_weights, start_node)

        # Check if a path was found to the sink and no negative cycle was detected (N/A with original costs).
        if dist is not None and dist[sink] != float('inf'):
            # Reconstruct the path as a list of arcs.
            path_arcs = reconstruct_path_arcs(pred_arc, start_node, sink)

            if path_arcs:
                initial_paths_arcs[k][start_node] = path_arcs # Store the found path arcs.
              

# Add initial path variables (columns) to the RMP based on the identified initial paths.
initial_rmp_vars = []
print("Adding initial RMP columns...")
for k in K:
    for o in origins:
        if o in initial_paths_arcs[k]:
            path_arcs = initial_paths_arcs[k][o]
            var_name = f"Path_{o}_{k}_Initial"
            # Add the path variable using the helper function.
            var = add_path_var(model, cap_con, sup_con, path_arcs, o, k, var_name, c)
            initial_rmp_vars.append(var) # Store the variable object


# Set the objective sense to minimization (already implicitly done by obj in addVar, but explicit is clear).
model.ModelSense = GRB.MINIMIZE # Correct way to set objective sense

# Update the model after adding initial variables and setting up constraints.
model.update()
print("Initial RMP built and updated.")


# --- Column Generation Loop ---
print("\nStarting Column Generation Loop...")

iteration = 0

min_reduced_cost_overall = -1.0 # Start with a negative value to enter the loop.


max_iterations = 100 # Increased iteration limit, but should converge much faster for this problem.


# Loop continues as long as negative reduced cost paths are found AND max iterations are not reached.

while min_reduced_cost_overall < -1e-6 and iteration < max_iterations:
    iteration += 1
    print(f"\n--- Iteration {iteration} ---")

    # Step 2: Solve the current Restricted Master Problem (RMP).
    model.optimize()

    # Check the optimization status of the RMP. If not optimal, something went wrong.
    if model.status != GRB.OPTIMAL:
        print(f"RMP not optimal. Status: {model.status}. Exiting column generation.")
        if model.status == GRB.INFEASIBLE:
             print("The RMP is infeasible.")
        break # Exit the main column generation loop

    # Step 3: Get the dual variables from the RMP solution.
    # Duals for the supply equality constraints. Access using .Pi attribute.
    
    sigma = {(o,k): sup_con[(o,k)].Pi for o in origins for k in K}
    
    # Duals for the capacity. These should be non-negative.
    alpha = {a: cap_con[a].Pi for a in A} # Capacity duals

    # Print the current RMP objective value and some duals for monitoring progress.
    print(f"RMP obj = {model.ObjVal:.4f}")
    # Print duals (sigma for supplies and non-zero alpha for capacities).
    print("  sigma (supplies):", {k: (sigma[(1,k)], sigma[(2,k)]) for k in K})
    print("  alpha (arcs with non-zero duals):", {a: round(alpha[a],6) for a in A if abs(alpha[a])>1e-8}) # Using 1e-8 tolerance 
    # Step 4: Find columns (paths) with negative reduced cost.
   
    new_columns_to_add = [] 
    min_reduced_cost_overall = 0 # Reset the minimum reduced cost found in this pricing step.
                                
    print("Solving pricing subproblems...")

    # Solve a pricing problem for each origin and commodity.
    for o in origins:
        for k in K:
            # Define the modified arc costs for the shortest path problem: original cost - capacity dual.
            w = {a: c[a] - alpha[a]*p[k][a] for a in A}


            dist, pred_arc = column_generation(nodes, A, w, o)

            if dist is None:
                 print(f"  Pricing (o={o}, k={k}): Negative cycle detected!")
                 print("  The RMP is unbounded. This suggests the original problem is unbounded.")
                 min_reduced_cost_overall = -float('inf') # Set to negative infinity to signal unboundedness.
                 break # Exit pricing loops

            # Check if the sink is reachable from the start node.
            if dist[sink] != float('inf'):
                # The shortest path cost found is the sum of the modified costs along the path.
                shortest_path_modified_cost = dist[sink]

                # Calculate the reduced cost of the found path.
                # Reduced Cost = (Shortest path cost on modified arcs) - (Supply dual for commodity and origin)
                red_cost = shortest_path_modified_cost - sigma[(o,k)]

                # Print the reduced cost found for each pricing subproblem.
                print(f"  Pricing (o={o}, k={k})  rc = {red_cost:.4f}") 

                # Step 5: Check if the reduced cost is negative.
                if red_cost < -1e-6:
                    # If the reduced cost is negative, the found path is a candidate for adding as a column.

                    # Update the minimum reduced cost found across all pricing problems in this iteration.
                    # This value is used for the overall convergence check.
                    min_reduced_cost_overall = min(min_reduced_cost_overall, red_cost)

                    # Reconstruct the path as a list of arcs.
                    path_arcs = reconstruct_path_arcs(pred_arc, o, sink)

                    # Ensure the path is valid before adding (check if reconstruction was successful).
                    if path_arcs is not None:
                         # Store the data needed to add this column later.
                         new_columns_to_add.append({'commodity': k, 'origin': o, 'path_arcs': path_arcs, 'reduced_cost': red_cost})
            
                    # else: print(f"    Pricing (o={o}, k={k}): Could not reconstruct path for a negative reduced cost column.")


        if min_reduced_cost_overall == -float('inf'):
             # If unboundedness detected in pricing for any commodity/source, stop all pricing.
             break

    # Print the minimum reduced cost found across all pricing problems in this iteration.
    print(f"  Minimum reduced cost found in pricing: {min_reduced_cost_overall:.4f}")

    # Step 6: Check stopping criterion.
    # If the minimum reduced cost found in this iteration is non-negative, no beneficial columns were found, and the algorithm converges.
    if min_reduced_cost_overall >= -1e-6:
        print("\nNo column with negative reduced cost found in pricing. Optimal solution reached.")
        break # Exit the main column generation loop.

    # Step 7: Add entering columns to the RMP.
    if new_columns_to_add:
        print(f"\nAdding {len(new_columns_to_add)} new columns to RMP...")
        # Iterate through the collected new column data and add each path variable to the RMP.
        for col_data in new_columns_to_add:
            k = col_data['commodity']
            o = col_data['origin']
            path_arcs = col_data['path_arcs']
            red_cost = col_data['reduced_cost'] 
            
            # Generate a unique name for the new path variable.
            name = f"Path_{o}_{k}_iter{iteration}_rc{red_cost:.2f}".replace('-', 'm').replace('.', 'd') # Make name valid chars
            added_var = add_path_var(model, cap_con, sup_con, path_arcs, o, k, name, c)

        # Important: Update the Gurobi model after adding variables and constraints in this iteration.
        model.update()
        print("RMP updated with new columns.")



print("\n==== FINAL SOLUTION ====")


# Check the final status of the model to ensure it's optimal.
if model.status == GRB.OPTIMAL and min_reduced_cost_overall >= -1e-6:
    # Print the optimal objective value of the RMP, which is the optimal value of the original problem.
    print(f"Optimal Total Cost = {model.ObjVal:.4f}")

    print("\nNon-zero Path Flows:")

    for v in model.getVars():
        if v.X > 1e-6:
            print(f"{v.VarName:30s}: {v.X:.4f}  (cost/unit = {v.Obj:.0f})")


Building initial RMP...
Finding initial paths (shortest paths on original costs)...
Adding initial RMP columns...
Initial RMP built and updated.

Starting Column Generation Loop...

--- Iteration 1 ---
RMP not optimal. Status: 3. Exiting column generation.
The RMP is infeasible.

==== FINAL SOLUTION ====


# 3. Generalized Assignment Problem (GAP)

## gurobi solution

In [99]:
import gurobipy as gp
from gurobipy import GRB

# Problem data
workers = range(10)  # i = 0 to 9
products = range(5)   # j = 0 to 4

a = [
    [3, 24, 53, 27, 17],
    [15, 23, 43, 74, 23],
    [54, 43, 27, 21, 36],
    [92, 83, 45, 35, 26],
    [19, 10, 33, 43, 12],
    [91, 55, 32, 26, 23],
    [15, 25, 36, 37, 28],
    [47, 43, 33, 28, 23],
    [34, 25, 32, 46, 43],
    [35, 23, 34, 25, 40]
]

c = [
    [15, 44, 76, 43, 34],
    [19, 23, 45, 46, 34],
    [10, 6, 3, 23, 15],
    [60, 45, 34, 36, 23],
    [11, 12, 34, 44, 10],
    [67, 65, 34, 20, 37],
    [23, 34, 24, 47, 27],
    [23, 25, 35, 15, 27],
    [12, 13, 24, 25, 24],
    [10, 15, 23, 12, 13]
]

b = [80, 63, 75, 98, 59, 87, 78, 90, 85, 82]

# Create a new model
model = gp.Model("Generalized Assignment Problem")

# Create variables
x = model.addVars(workers, products, vtype=GRB.BINARY, name="x")

# Set objective function: Maximize total profit
model.setObjective(gp.quicksum(c[i][j] * x[i, j] for i in workers for j in products), GRB.MAXIMIZE)

# Add constraints

# Each product is assigned to exactly one worker
for j in products:
    model.addConstr(gp.quicksum(x[i, j] for i in workers) == 1, f"ProductAssignment_{j}")

# Total resource consumption for each worker does not exceed available resource
for i in workers:
    model.addConstr(gp.quicksum(a[i][j] * x[i, j] for j in products) <= b[i], f"WorkerResource_{i}")

# Optimize the model
model.optimize()

# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    print(f"Optimal Total Profit: {model.objVal}")
    print("\nAssignments:")
    for i in workers:
        for j in products:
            if x[i, j].x > 0.5:  # Check if the variable is essentially 1
                print(f"  Worker {i+1} is assigned to Product {j+1}")
else:
    print("No optimal solution found.")
    print(f"Optimization status: {model.status}")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 15 rows, 50 columns and 100 nonzeros
Model fingerprint: 0x86534e00
Variable types: 0 continuous, 50 integer (50 binary)
Coefficient statistics:
  Matrix range     [1e+00, 9e+01]
  Objective range  [3e+00, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Found heuristic solution: objective 97.0000000
Presolve removed 0 rows and 8 columns
Presolve time: 0.01s
Presolved: 15 rows, 42 columns, 84 nonzeros
Found heuristic solution: objective 285.0000000
Variable types: 0 continuous, 42 integer (42 binary)

Root relaxation: cutoff, 17 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0     cutoff    0       285.00

## LP Relaxation

In [101]:
import gurobipy as gp
from gurobipy import GRB

# Problem data (same as before)
workers = range(10)  # i = 0 to 9
products = range(5)   # j = 0 to 4

a = [
    [3, 24, 53, 27, 17],
    [15, 23, 43, 74, 23],
    [54, 43, 27, 21, 36],
    [92, 83, 45, 35, 26],
    [19, 10, 33, 43, 12],
    [91, 55, 32, 26, 23],
    [15, 25, 36, 37, 28],
    [47, 43, 33, 28, 23],
    [34, 25, 32, 46, 43],
    [35, 23, 34, 25, 40]
]

c = [
    [15, 44, 76, 43, 34],
    [19, 23, 45, 46, 34],
    [10, 6, 3, 23, 15],
    [60, 45, 34, 36, 23],
    [11, 12, 34, 44, 10],
    [67, 65, 34, 20, 37],
    [23, 34, 24, 47, 27],
    [23, 25, 35, 15, 27],
    [12, 13, 24, 25, 24],
    [10, 15, 23, 12, 13]
]

b = [80, 63, 75, 98, 59, 87, 78, 90, 85, 82]

# Create a new model for the LP relaxation
lp_model = gp.Model("Generalized Assignment Problem - LP Relaxation")

# Create variables (continuous between 0 and 1)
# By default, Gurobi variables are continuous with bounds [0, infinity].
# We need to explicitly set the upper bound to 1.
x_lp = lp_model.addVars(workers, products, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="x_lp")

# Set objective function: Maximize total profit
lp_model.setObjective(gp.quicksum(c[i][j] * x_lp[i, j] for i in workers for j in products), GRB.MAXIMIZE)

# Add constraints (same as MIP)

# Each product is assigned to exactly one worker
for j in products:
    lp_model.addConstr(gp.quicksum(x_lp[i, j] for i in workers) == 1, f"ProductAssignment_{j}")

# Total resource consumption for each worker does not exceed available resource
for i in workers:
    lp_model.addConstr(gp.quicksum(a[i][j] * x_lp[i, j] for j in products) <= b[i], f"WorkerResource_{i}")

# Optimize the LP relaxation
lp_model.optimize()

# Print the results of the LP relaxation
if lp_model.status == GRB.OPTIMAL:
    print("Optimal solution for the initial LP Relaxation found:")
    print(f"LP Relaxation Objective Value (Upper Bound): {lp_model.objVal}")
    print("\nLP Relaxation Variable Values:")
    for i in workers:
        for j in products:
            if x_lp[i, j].x > 1e-6: # Print non-zero values
                print(f"  x_{i+1},{j+1} = {x_lp[i, j].x}")
else:
    print("LP Relaxation did not find an optimal solution.")
    print(f"Optimization status: {lp_model.status}")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 15 rows, 50 columns and 100 nonzeros
Model fingerprint: 0x3fcaad9b
Coefficient statistics:
  Matrix range     [1e+00, 9e+01]
  Objective range  [3e+00, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve time: 0.01s
Presolved: 15 rows, 50 columns, 100 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.1923095e+02   7.460477e+01   0.000000e+00      0s
      10    2.8569231e+02   0.000000e+00   0.000000e+00      0s

Solved in 10 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.856923077e+02
Optimal solution for the initial LP Relaxation found:
LP Relaxation Objective Value (Upper Bound): 285.69230769230774

LP Relaxation Variable Values:
  x_1,3 = 1.0
  x_4,1 = 0.9010989010989011
  x_6,1 = 0.0989010989

### iteration 2

In [103]:
import gurobipy as gp
from gurobipy import GRB

# Problem data (same as before)
workers = range(10)  # i = 0 to 9
products = range(5)   # j = 0 to 4

a = [
    [3, 24, 53, 27, 17],
    [15, 23, 43, 74, 23],
    [54, 43, 27, 21, 36],
    [92, 83, 45, 35, 26],
    [19, 10, 33, 43, 12],
    [91, 55, 32, 26, 23],
    [15, 25, 36, 37, 28],
    [47, 43, 33, 28, 23],
    [34, 25, 32, 46, 43],
    [35, 23, 34, 25, 40]
]

c = [
    [15, 44, 76, 43, 34],
    [19, 23, 45, 46, 34],
    [10, 6, 3, 23, 15],
    [60, 45, 34, 36, 23],
    [11, 12, 34, 44, 10],
    [67, 65, 34, 20, 37],
    [23, 34, 24, 47, 27],
    [23, 25, 35, 15, 27],
    [12, 13, 24, 25, 24],
    [10, 15, 23, 12, 13]
]

b = [80, 63, 75, 98, 59, 87, 78, 90, 85, 82]

# Create a new model for Subproblem 1.1 (LP Relaxation with x[3,0] = 0)
model_1_1 = gp.Model("GAP_LP_Subproblem_1_1_x_3_0_eq_0")

# Create variables (continuous between 0 and 1)
x_1_1 = model_1_1.addVars(workers, products, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="x")

# Set objective function: Maximize total profit
model_1_1.setObjective(gp.quicksum(c[i][j] * x_1_1[i, j] for i in workers for j in products), GRB.MAXIMIZE)

# Add original constraints

# Each product is assigned to exactly one worker
for j in products:
    model_1_1.addConstr(gp.quicksum(x_1_1[i, j] for i in workers) == 1, f"ProductAssignment_{j}")

# Total resource consumption for each worker does not exceed available resource
for i in workers:
    model_1_1.addConstr(gp.quicksum(a[i][j] * x_1_1[i, j] for j in products) <= b[i], f"WorkerResource_{i}")

# Add the branching constraint for Subproblem 1.1: x[3, 0] = 0 (Worker 4, Product 1)
model_1_1.addConstr(x_1_1[3, 0] == 0, "Branch_x_3_0_eq_0")


# Optimize the model
model_1_1.optimize()

# Print the results of Subproblem 1.1 LP Relaxation
print("\n--- Results for Subproblem 1.1 (x[3,0] = 0) ---")
if model_1_1.status == GRB.OPTIMAL:
    print("Optimal solution for LP Relaxation of Subproblem 1.1 found:")
    print(f"LP Relaxation Objective Value: {model_1_1.objVal}")
    print("\nLP Relaxation Variable Values:")
    for i in workers:
        for j in products:
            if x_1_1[i, j].x > 1e-6: # Print non-zero values
                # Use 1-based indexing for printing to match problem description
                print(f"  x_{i+1},{j+1} = {x_1_1[i, j].x}")
elif model_1_1.status == GRB.INF_OR_UNBD or model_1_1.status == GRB.INFEASIBLE:
     print("Subproblem 1.1 LP Relaxation is Infeasible.")
     # You might want to run feasibility relaxation here in a real scenario
elif model_1_1.status == GRB.UNBOUNDED:
    print("Subproblem 1.1 LP Relaxation is Unbounded.")
else:
    print("Subproblem 1.1 LP Relaxation did not find an optimal solution.")
    print(f"Optimization status: {model_1_1.status}")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 50 columns and 101 nonzeros
Model fingerprint: 0x95e4bf72
Coefficient statistics:
  Matrix range     [1e+00, 9e+01]
  Objective range  [3e+00, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 1 rows and 1 columns
Presolve time: 0.01s
Presolved: 15 rows, 49 columns, 98 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.5250391e+02   7.197632e+01   0.000000e+00      0s
      10    2.6706593e+02   0.000000e+00   0.000000e+00      0s

Solved in 10 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.670659341e+02

--- Results for Subproblem 1.1 (x[3,0] = 0) ---
Optimal solution for LP Relaxation of Subproblem 1.1 found:
LP Relaxation Objective Value: 267.0659340659341

LP Relaxation Varia

### iteration 3

In [127]:
import gurobipy as gp
from gurobipy import GRB

# Problem data (same as before)
workers = range(10)  # i = 0 to 9
products = range(5)   # j = 0 to 4

a = [
    [3, 24, 53, 27, 17],
    [15, 23, 43, 74, 23],
    [54, 43, 27, 21, 36],
    [92, 83, 45, 35, 26],
    [19, 10, 33, 43, 12],
    [91, 55, 32, 26, 23],
    [15, 25, 36, 37, 28],
    [47, 43, 33, 28, 23],
    [34, 25, 32, 46, 43],
    [35, 23, 34, 25, 40]
]

c = [
    [15, 44, 76, 43, 34],
    [19, 23, 45, 46, 34],
    [10, 6, 3, 23, 15],
    [60, 45, 34, 36, 23],
    [11, 12, 34, 44, 10],
    [67, 65, 34, 20, 37],
    [23, 34, 24, 47, 27],
    [23, 25, 35, 15, 27],
    [12, 13, 24, 25, 24],
    [10, 15, 23, 12, 13]
]

b = [80, 63, 75, 98, 59, 87, 78, 90, 85, 82]

# Create a new model for Subproblem 1.1 (LP Relaxation with x[3,0] = 0)
model_1_1 = gp.Model("GAP_LP_Subproblem_1_1_x_3_0_eq_0")

# Create variables (continuous between 0 and 1)
x_1_1 = model_1_1.addVars(workers, products, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="x")

# Set objective function: Maximize total profit
model_1_1.setObjective(gp.quicksum(c[i][j] * x_1_1[i, j] for i in workers for j in products), GRB.MAXIMIZE)

# Add original constraints

# Each product is assigned to exactly one worker
for j in products:
    model_1_1.addConstr(gp.quicksum(x_1_1[i, j] for i in workers) == 1, f"ProductAssignment_{j}")

# Total resource consumption for each worker does not exceed available resource
for i in workers:
    model_1_1.addConstr(gp.quicksum(a[i][j] * x_1_1[i, j] for j in products) <= b[i], f"WorkerResource_{i}")

# Add the branching constraint for Subproblem 1.1: x[3, 0] = 0 (Worker 4, Product 1)
model_1_1.addConstr(x_1_1[3, 0] == 1, "Branch_x_3_0_eq_1")


# Optimize the model
model_1_1.optimize()

# Print the results of Subproblem 1.1 LP Relaxation
print("\n--- Results for Subproblem 1.1 (x[3,0] = 1) ---")
if model_1_1.status == GRB.OPTIMAL:
    print("Optimal solution for LP Relaxation of Subproblem 1.1 found:")
    print(f"LP Relaxation Objective Value: {model_1_1.objVal}")
    print("\nLP Relaxation Variable Values:")
    for i in workers:
        for j in products:
            if x_1_1[i, j].x > 1e-6: # Print non-zero values
                # Use 1-based indexing for printing to match problem description
                print(f"  x_{i+1},{j+1} = {x_1_1[i, j].x}")
elif model_1_1.status == GRB.INF_OR_UNBD or model_1_1.status == GRB.INFEASIBLE:
     print("Subproblem 1.1 LP Relaxation is Infeasible.")
     # You might want to run feasibility relaxation here in a real scenario
elif model_1_1.status == GRB.UNBOUNDED:
    print("Subproblem 1.1 LP Relaxation is Unbounded.")
else:
    print("Subproblem 1.1 LP Relaxation did not find an optimal solution.")
    print(f"Optimization status: {model_1_1.status}")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 50 columns and 101 nonzeros
Model fingerprint: 0x5c37ba17
Coefficient statistics:
  Matrix range     [1e+00, 9e+01]
  Objective range  [3e+00, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 2 rows and 10 columns
Presolve time: 0.01s
Presolved: 14 rows, 40 columns, 80 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.8500000e+02   0.000000e+00   0.000000e+00      0s
       0    2.8500000e+02   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Optimal objective  2.850000000e+02

--- Results for Subproblem 1.1 (x[3,0] = 1) ---
Optimal solution for LP Relaxation of Subproblem 1.1 found:
LP Relaxation Objective Value: 285.0

LP Relaxation Variable Values:


## b 

In [123]:
import gurobipy as gp
from gurobipy import GRB

# Problem data (same as before)
workers = range(10)  # i = 0 to 9
products = range(5)   # j = 0 to 4

a = [
    [3, 24, 53, 27, 17],
    [15, 23, 43, 74, 23],
    [54, 43, 27, 21, 36],
    [92, 83, 45, 35, 26],
    [19, 10, 33, 43, 12],
    [91, 55, 32, 26, 23],
    [15, 25, 36, 37, 28],
    [47, 43, 33, 28, 23],
    [34, 25, 32, 46, 43],
    [35, 23, 34, 25, 40]
]

c = [
    [15, 44, 76, 43, 34],
    [19, 23, 45, 46, 34],
    [10, 6, 3, 23, 15],
    [60, 45, 34, 36, 23],
    [11, 12, 34, 44, 10],
    [67, 65, 34, 20, 37],
    [23, 34, 24, 47, 27],
    [23, 25, 35, 15, 27],
    [12, 13, 24, 25, 24],
    [10, 15, 23, 12, 13]
]

b = [80, 63, 75, 98, 59, 87, 78, 90, 85, 82]

# Create a new model for the LP relaxation with multiple cutting planes
lp_model_multiple_cuts = gp.Model("GAP_LP_Relaxation_Multiple_Cuts")

# Create variables (continuous between 0 and 1)
x_multiple_cuts = lp_model_multiple_cuts.addVars(workers, products, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="x")

# Set objective function: Maximize total profit
lp_model_multiple_cuts.setObjective(gp.quicksum(c[i][j] * x_multiple_cuts[i, j] for i in workers for j in products), GRB.MAXIMIZE)

# Add original constraints

# Each product is assigned to exactly one worker
for j in products:
    lp_model_multiple_cuts.addConstr(gp.quicksum(x_multiple_cuts[i, j] for i in workers) == 1, f"ProductAssignment_{j}")

# Total resource consumption for each worker does not exceed available resource
for i in workers:
    lp_model_multiple_cuts.addConstr(gp.quicksum(a[i][j] * x_multiple_cuts[i, j] for j in products) <= b[i], f"WorkerResource_{i}")

# Add Multiple Cover Inequalities (Valid Inequalities)

# Cover inequality for Worker 0 (index 0), Products 2, 3, 4 (indices 1, 2, 3)
# x[0,1] + x[0,2] + x[0,3] <= 2
lp_model_multiple_cuts.addConstr(x_multiple_cuts[0, 1] + x_multiple_cuts[0, 2] + x_multiple_cuts[0, 3] <= 2, "Cover_W1_P2_P3_P4")

# Cover inequality for Worker 1 (index 1), Product 4 (index 3)
# x[1,3] == 0
lp_model_multiple_cuts.addConstr(x_multiple_cuts[1, 3] == 0, "Cover_W2_P4")

# Cover inequality for Worker 2 (index 2), Products 1, 2 (indices 0, 1)
# x[2,0] + x[2,1] <= 1
lp_model_multiple_cuts.addConstr(x_multiple_cuts[2, 0] + x_multiple_cuts[2, 1] <= 1, "Cover_W3_P1_P2")


# Optimize the model
lp_model_multiple_cuts.optimize()

# Print the results of the LP relaxation with cuts
print("\n--- Results for the LP Relaxation with Multiple Cover Inequalities ---")
if lp_model_multiple_cuts.status == GRB.OPTIMAL:
    print("Optimal solution for LP Relaxation with multiple cuts found:")
    print(f"LP Relaxation Objective Value: {lp_model_multiple_cuts.objVal}")
    print("\nLP Relaxation Variable Values:")
    for i in workers:
        for j in products:
            if x_multiple_cuts[i, j].x > 1e-6: # Print non-zero values
                # Use 1-based indexing for printing
                print(f"  x_{i+1},{j+1} = {x_multiple_cuts[i, j].x}")
elif lp_model_multiple_cuts.status == GRB.INF_OR_UNBD or lp_model_multiple_cuts.status == GRB.INFEASIBLE:
     print("LP Relaxation with multiple cuts is Infeasible.")
elif lp_model_multiple_cuts.status == GRB.UNBOUNDED:
    print("LP Relaxation with multiple cuts is Unbounded.")
else:
    print("LP Relaxation with multiple cuts did not find an optimal solution.")
    print(f"Optimization status: {lp_model_multiple_cuts.status}")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 18 rows, 50 columns and 106 nonzeros
Model fingerprint: 0xfdc22b9f
Coefficient statistics:
  Matrix range     [1e+00, 9e+01]
  Objective range  [3e+00, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 1 rows and 1 columns
Presolve time: 0.00s
Presolved: 17 rows, 49 columns, 103 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.1641695e+02   7.719613e+01   0.000000e+00      0s
      11    2.8569231e+02   0.000000e+00   0.000000e+00      0s

Solved in 11 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.856923077e+02

--- Results for the LP Relaxation with Multiple Cover Inequalities ---
Optimal solution for LP Relaxation with multiple cuts found:
LP Relaxation Objective Value: 285.692307692

In [170]:
import gurobipy as gp
from gurobipy import GRB

# Problem data (same as before)
workers = range(10)  # i = 0 to 9
products = range(5)   # j = 0 to 4

a = [
    [3, 24, 53, 27, 17],
    [15, 23, 43, 74, 23],
    [54, 43, 27, 21, 36],
    [92, 83, 45, 35, 26],
    [19, 10, 33, 43, 12],
    [91, 55, 32, 26, 23],
    [15, 25, 36, 37, 28],
    [47, 43, 33, 28, 23],
    [34, 25, 32, 46, 43],
    [35, 23, 34, 25, 40]
]

c = [
    [15, 44, 76, 43, 34],
    [19, 23, 45, 46, 34],
    [10, 6, 3, 23, 15],
    [60, 45, 34, 36, 23],
    [11, 12, 34, 44, 10],
    [67, 65, 34, 20, 37],
    [23, 34, 24, 47, 27],
    [23, 25, 35, 15, 27],
    [12, 13, 24, 25, 24],
    [10, 15, 23, 12, 13]
]

b = [80, 63, 75, 98, 59, 87, 78, 90, 85, 82]

# Create a new model for the LP relaxation with multiple cutting planes
lp_model_multiple_cuts = gp.Model("GAP_LP_Relaxation_Multiple_Cuts")

# Create variables (continuous between 0 and 1)
x_multiple_cuts = lp_model_multiple_cuts.addVars(workers, products, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="x")

# Set objective function: Maximize total profit
lp_model_multiple_cuts.setObjective(gp.quicksum(c[i][j] * x_multiple_cuts[i, j] for i in workers for j in products), GRB.MAXIMIZE)

# Add original constraints

# Each product is assigned to exactly one worker
for j in products:
    lp_model_multiple_cuts.addConstr(gp.quicksum(x_multiple_cuts[i, j] for i in workers) == 1, f"ProductAssignment_{j}")

# Total resource consumption for each worker does not exceed available resource
for i in workers:
    lp_model_multiple_cuts.addConstr(gp.quicksum(a[i][j] * x_multiple_cuts[i, j] for j in products) <= b[i], f"WorkerResource_{i}")

# Add Multiple Cover Inequalities (Valid Inequalities)

# Cover inequality for Worker 0 (index 0), Products 2, 3, 4 (indices 1, 2, 3)
# x[0,1] + x[0,2] + x[0,3] <= 2
lp_model_multiple_cuts.addConstr(x_multiple_cuts[0, 1] + x_multiple_cuts[0, 2] + x_multiple_cuts[0, 3] <= 2, "Cover_W1_P2_P3_P4")

# Cover inequality for Worker 1 (index 1), Product 4 (index 3)
# x[1,3] == 0
lp_model_multiple_cuts.addConstr(x_multiple_cuts[1, 3] == 0, "Cover_W2_P4")

# Cover inequality for Worker 2 (index 2), Products 1, 2 (indices 0, 1)
# x[2,0] + x[2,1] <= 1
lp_model_multiple_cuts.addConstr(x_multiple_cuts[2, 0] + x_multiple_cuts[2, 1] <= 1, "Cover_W3_P1_P2")

# Add the branching constraint for Subproblem 1.1: x[3, 0] = 0 (Worker 4, Product 1)
lp_model_multiple_cuts.addConstr(x_multiple_cuts[3, 0] == 1, "Branch_x_3_0_eq_0")

# Optimize the model
lp_model_multiple_cuts.optimize()

# Print the results of the LP relaxation with cuts
print("\n--- Results for the LP Relaxation with Multiple Cover Inequalities ---")
if lp_model_multiple_cuts.status == GRB.OPTIMAL:
    print("Optimal solution for LP Relaxation with multiple cuts found:")
    print(f"LP Relaxation Objective Value: {lp_model_multiple_cuts.objVal}")
    print("\nLP Relaxation Variable Values:")
    for i in workers:
        for j in products:
            
            if x_multiple_cuts[i, j].x > 1e-6: # Print non-zero values
                # Use 1-based indexing for printing
                print(f"  x_{i+1},{j+1} = {x_multiple_cuts[i, j].x}")
elif lp_model_multiple_cuts.status == GRB.INF_OR_UNBD or lp_model_multiple_cuts.status == GRB.INFEASIBLE:
     print("LP Relaxation with multiple cuts is Infeasible.")
elif lp_model_multiple_cuts.status == GRB.UNBOUNDED:
    print("LP Relaxation with multiple cuts is Unbounded.")
else:
    print("LP Relaxation with multiple cuts did not find an optimal solution.")
    print(f"Optimization status: {lp_model_multiple_cuts.status}")

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G93)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 19 rows, 50 columns and 107 nonzeros
Model fingerprint: 0xe051d59b
Coefficient statistics:
  Matrix range     [1e+00, 9e+01]
  Objective range  [3e+00, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 4 rows and 11 columns
Presolve time: 0.00s
Presolved: 15 rows, 39 columns, 81 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.8500000e+02   0.000000e+00   0.000000e+00      0s
       0    2.8500000e+02   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.850000000e+02

--- Results for the LP Relaxation with Multiple Cover Inequalities ---
Optimal solution for LP Relaxation with multiple cuts found:
LP Relaxation Objective Value: 285.0

LP Rela